In [63]:
import sys
sys.path.append('../')
import numpy as np
import matplotlib.pyplot as plt
import os
from pydicom import dcmread

from types import MappingProxyType

from pydose_rt import ModelConfig
from models.model import *
from models.layers import *
from engine.data import DataGenerator
# from models.data import DataGenerator
from models.utils import *
from models.kernel import *

from engine.config import config as PARAMS
import engine.utils.path_utils as path_utils

In [64]:
import os
cwd = os.getcwd()
cwd

'/mnt/SSD/github/autoplan/notebooks'

In [65]:
data_path = path_utils.get_parent_dir(cwd) + "/database/AUTORPT/"

from engine.config import config as PARAMS

gen = DataGenerator(
    data_path,
    "training",
    True,
    1,
    constraints=PARAMS.constraints,
    is_debug=0,
    is_return_gaussian_ptv_oars=True,
)

Number of files: 1092 in training cohort


In [66]:
def analyze_oar_mask_z_extent(array_5d):
    """
    Analyzes a 5D NumPy array to find the z-extent of combined OAR masks.

    Assumes the array has shape (1, height, width, depth, channels), where:
    - channels dimension (last dimension) contains binary masks for OARs at indices 1 to 5.
    - z-axis is the third dimension (index 2, size 'depth').

    Args:
        array_5d (numpy.ndarray): A 5D NumPy array with shape (1, height, width, depth, channels),
                                   where channels >= 6.

    Returns:
        tuple: A tuple containing (min_z, max_z, z_extent), where:
               - min_z (int or None): Minimum z-index where OAR mask is True, or None if no OAR is present.
               - max_z (int or None): Maximum z-index where OAR mask is True, or None if no OAR is present.
               - z_extent (int or None): Difference between max_z and min_z, or None if no OAR is present.
    """

    if array_5d.ndim != 5:
        raise ValueError("Input array must be 5-dimensional.")

    if array_5d.shape[0] != 1:
        raise ValueError("First dimension of input array must be 1.")

    if array_5d.shape[-1] < 6:
        raise ValueError("Last dimension of input array must have size at least 6 to contain OAR masks (indices 1-5).")

    # 1. Remove the first dimension to get 4D array (height, width, depth, channels)
    array_4d = np.squeeze(array_5d, axis=0)

    # 2. Extract OAR binary masks (indices 1 to 5 in the last dimension) and sum them
    oar_masks = array_4d[..., 0:6] # Slicing to get indices 1, 2, 3, 4, 5 (exclusive of 6)
    combined_oar_mask = np.sum(oar_masks, axis=-1) # Sum along the last dimension (channels)

    # Now combined_oar_mask has shape (128, 128, 320) - assuming input height=128, width=128, depth=320

    # 3. Initialize min_z and max_z to None, assuming no OAR initially
    min_z = None
    max_z = None

    # 4. Iterate along the z-axis (depth, index 2)
    depth_size = combined_oar_mask.shape[2] # Size of the z-dimension (320)
    found_oar = False # Flag to check if any OAR is found

    for z_index in range(depth_size):
        z_slice = combined_oar_mask[:, :, z_index] # Extract a slice along the z-axis

        if np.any(z_slice): # Check if any element in the slice is True (non-zero)
            found_oar = True
            if min_z is None:
                min_z = z_index # First z-slice with OAR, so set min_z
            max_z = z_index # Update max_z as we find OAR in this z-slice

    # 5. Calculate z_extent if OARs were found
    if found_oar:
        z_extent = max_z - min_z
    else:
        min_z = None # Reset to None if no OAR found in any slice
        max_z = None
        z_extent = None

    return min_z, max_z, z_extent

In [67]:
def analyze_nonzero_values(gaussian_ptv_oars):
    """
    Analyze non-zero values in a NumPy array.
    
    Args:
        gaussian_ptv_oars: NumPy array to analyze
        
    Returns:
        tuple: (min_nonzero, count_nonzero, percentage_nonzero)
            - min_nonzero: Minimum non-zero value in the array
            - count_nonzero: Number of non-zero elements
            - percentage_nonzero: Percentage of non-zero elements (0-100)
    """
    # Get total number of elements
    total_elements = gaussian_ptv_oars.size
    
    # Create a mask for non-zero elements
    nonzero_mask = gaussian_ptv_oars != 0
    
    # Count non-zero elements
    count_nonzero = np.count_nonzero(nonzero_mask)
    
    # Calculate percentage of non-zero elements
    percentage_nonzero = (count_nonzero / total_elements) * 100
    
    # Find minimum non-zero value
    if count_nonzero > 0:
        # Create a masked array to find min of only non-zero values
        masked_array = np.ma.masked_where(~nonzero_mask, gaussian_ptv_oars)
        min_nonzero = np.ma.min(masked_array)
    else:
        min_nonzero = None  # No non-zero values
    
    return min_nonzero, count_nonzero, percentage_nonzero

In [68]:
def compute_percent(gaussian_ptv_oars, masks):
    _, count_gaussian_ptv_oars, _ = analyze_nonzero_values(gaussian_ptv_oars)
    ptv_oar_masks = masks[..., 0:6] # Slicing to get indices 1, 2, 3, 4, 5 (exclusive of 6)
    combined_oar_mask = np.sum(ptv_oar_masks, axis=-1) # Sum along the last dimension (channels)
    _, count_combined_oar_mask, _ = analyze_nonzero_values(combined_oar_mask)
    _, count_background, _ = analyze_nonzero_values(masks[..., -1])
    
    # print(count_gaussian_ptv_oars,count_combined_oar_mask,count_background)
    
    percentage_nonzero = ((count_gaussian_ptv_oars - count_combined_oar_mask) / count_background) * 100    
    return percentage_nonzero


In [69]:

list_min, list_max, list_depth, list_percent = [], [], [], []

for i in range(len(gen)):
    x, y, masks, _, gaussian_ptv_oars = gen[i]
            
    min_z, max_z, z_extent = analyze_oar_mask_z_extent(masks)
    percentage_nonzero = compute_percent(gaussian_ptv_oars, masks)
    percentage_nonzero = round(percentage_nonzero,2)
    
    list_min.append(min_z)
    list_max.append(max_z)
    list_depth.append(z_extent)
    list_percent.append(percentage_nonzero)
    print(f"{i} \t {min_z} \t {max_z} \t {z_extent} \t {percentage_nonzero}")
    

0 	 108 	 210 	 102 	 1.38
1 	 108 	 211 	 103 	 1.23
2 	 112 	 267 	 155 	 2.61
3 	 119 	 200 	 81 	 1.32
4 	 114 	 303 	 189 	 3.27
5 	 123 	 208 	 85 	 1.45
6 	 102 	 268 	 166 	 2.62
7 	 102 	 215 	 113 	 2.22
8 	 120 	 219 	 99 	 1.5
9 	 112 	 272 	 160 	 2.82
10 	 97 	 212 	 115 	 1.4
11 	 117 	 292 	 175 	 2.9
12 	 113 	 212 	 99 	 1.4
13 	 114 	 288 	 174 	 2.7
14 	 107 	 216 	 109 	 1.36
15 	 93 	 221 	 128 	 1.51
16 	 105 	 217 	 112 	 2.2
17 	 113 	 226 	 113 	 1.61
18 	 105 	 202 	 97 	 1.27
19 	 88 	 308 	 220 	 2.95
20 	 107 	 226 	 119 	 1.48
21 	 92 	 209 	 117 	 1.87
22 	 123 	 208 	 85 	 1.45
23 	 102 	 268 	 166 	 2.62
24 	 105 	 226 	 121 	 1.61
25 	 119 	 258 	 139 	 1.89
26 	 119 	 228 	 109 	 1.56
27 	 119 	 228 	 109 	 1.56
28 	 107 	 226 	 119 	 1.48
29 	 121 	 220 	 99 	 1.5
30 	 102 	 213 	 111 	 1.64
31 	 112 	 206 	 94 	 1.29
32 	 90 	 214 	 124 	 2.78
33 	 83 	 216 	 133 	 2.21
34 	 117 	 199 	 82 	 1.19
35 	 112 	 272 	 160 	 2.82
36 	 103 	 253 	 150 	 2

KeyboardInterrupt: 